# Validation & Uncertainty Quantification — POC

This notebook explores how confident the U-Net seabed detector actually is. The model outputs a probability map, but so far we have only thresholded it at 0.5 and measured IoU/Dice. Here we look deeper: how stable are the predictions under dropout noise (MC-Dropout), how well-calibrated are the probabilities, and how the choice of threshold affects the results.

The ground truth itself is algorithmically generated (ensemble of simple + edge detectors), so quantifying prediction uncertainty is especially relevant — both the model and the labels carry their own noise.

In [ ]:
import sys
sys.path.insert(0, '../..')

import numpy as np
import matplotlib.pyplot as plt
import torch

from deep_learning.unet_model import create_unet
from deep_learning.evaluate import (
    evaluate_file, extract_bottom_line,
)
from validation.uncertainty import (
    mc_dropout_predict, prediction_entropy, bottom_uncertainty_band,
)
from validation.metrics import threshold_sensitivity, calibration_curve
from project_paths import REPO_ROOT

## Load checkpoint and data

In [ ]:
DATASETS = REPO_ROOT / 'Datasets'
ZARR_DIR = DATASETS / 'echopype_zarr'
CHECKPOINT = REPO_ROOT / 'code' / 'checkpoints' / 'exp_20260420_080950_finetune_stages' / 'unet_resnet34_best.pth'

device = 'cuda' if torch.cuda.is_available() else 'cpu'

# Load model
ckpt = torch.load(CHECKPOINT, map_location=device, weights_only=False)
cfg = ckpt.get('config', {})

model = create_unet(
    encoder_name=cfg.get('encoder_name', 'resnet34'),
    in_channels=3,
    classes=1,
)
if cfg.get('decoder_attention') == 'scse':
    import segmentation_models_pytorch as smp
    model = smp.Unet(
        encoder_name=cfg.get('encoder_name', 'resnet34'),
        encoder_weights=None,
        in_channels=3,
        classes=1,
        activation=None,
        decoder_attention_type='scse',
    )

model.load_state_dict(ckpt['model_state_dict'])
model = model.to(device)
model.eval()

target_size = tuple(cfg.get('target_size', (256, 256)))
freq1_hz = cfg.get('freq1_hz', 38_000)
freq2_hz = cfg.get('freq2_hz', 200_000)
vmin = cfg.get('vmin', -80.0)
vmax = cfg.get('vmax', -34.0)
stride = cfg.get('stride', 128)

print(f'Device: {device}')
print(f'Encoder: {cfg.get("encoder_name", "resnet34")}')
print(f'Target size: {target_size}')
print(f'Loaded from epoch {ckpt.get("epoch", "?")} — val IoU {ckpt.get("val_iou", "?")}')

In [ ]:
# Pick the first validation zarr file for analysis
zarr_paths = sorted(ZARR_DIR.glob('*.zarr'))
val_zarr = zarr_paths[0]
print(f'Using: {val_zarr.name}')

# Run standard evaluation to get prediction, GT, and input channels
metrics, prediction, gt_mask, sv_channels, range_arr = evaluate_file(
    model, val_zarr,
    freq1_hz=freq1_hz, freq2_hz=freq2_hz,
    target_size=target_size, device=device,
    stride=stride, vmin=vmin, vmax=vmax,
)

print(f'Prediction shape: {prediction.shape}')
print(f'IoU={metrics.iou:.4f}  Dice={metrics.dice:.4f}  '
      f'Bottom err: {metrics.bottom_error_mean_m:.2f}m (mean)')

## MC-Dropout uncertainty

MC-Dropout keeps the model's dropout layers active at inference time and runs multiple stochastic forward passes. Each pass produces a slightly different probability map because different neurons are randomly dropped. The mean across passes is a better-calibrated prediction, and the standard deviation indicates where the model is uncertain.

With a ResNet34 encoder the dropout is in the decoder path, so the effect is modest but still informative at the seabed boundary where small changes in features shift the predicted depth by a few pixels.

In [ ]:
import copy
from deep_learning.unet_model import add_mc_dropout

DROPOUT_P = 0.2
N_SAMPLES = 50

# Work on a fresh copy so the original model stays clean
model_mc = copy.deepcopy(model)
add_mc_dropout(model_mc, p=DROPOUT_P)

mean_prob, std_prob, all_preds = mc_dropout_predict(
    model, sv_channels, target_size, device,
    n_samples=50, patch_width=256, stride=stride,
    dropout_p=0.2,
)

# Sanity: confirm BN is in eval mode after MC inference
bn_train = [m.training for m in model_mc.modules() if isinstance(m, torch.nn.BatchNorm2d)]
print(f'BN layers in train mode during MC: {sum(bn_train)}/{len(bn_train)} '
      f'(should be 0)')
print(f'MC-Dropout samples: {all_preds.shape[0]}, dropout_p={DROPOUT_P}')
print(f'Mean prediction range: [{mean_prob.min():.4f}, {mean_prob.max():.4f}]')
print(f'Std range: [{std_prob.min():.6f}, {std_prob.max():.6f}]')
print(f'Max uncertainty at: {np.unravel_index(std_prob.argmax(), std_prob.shape)}')

In [ ]:
n_range, n_pings = mean_prob.shape
extent = [0, n_pings, range_arr[-1], range_arr[0]]

fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharex=True, sharey=True)

axes[0].imshow(sv_channels[0], aspect='auto', cmap='viridis', vmin=0, vmax=1,
               origin='upper', extent=extent)
axes[0].set_title('Echogram (38 kHz)')
axes[0].set_ylabel('Range (m)')

axes[1].imshow(mean_prob, aspect='auto', cmap='gray_r', vmin=0, vmax=1,
               origin='upper', extent=extent)
axes[1].set_title('MC-Dropout mean prediction')

im = axes[2].imshow(std_prob, aspect='auto', cmap='hot', origin='upper', extent=extent)
axes[2].set_title('MC-Dropout std (uncertainty)')
plt.colorbar(im, ax=axes[2], label='Std dev')

for ax in axes:
    ax.set_xlabel('Ping')

plt.tight_layout()
plt.show()

In [ ]:
# Compare MC-Dropout uncertainty across different dropout rates
dropout_rates = [0.1, 0.2, 0.3]
mc_results = {}

for p in dropout_rates:
    model_tmp = copy.deepcopy(model)
    add_mc_dropout(model_tmp, p=p)
    m_prob, s_prob, _ = mc_dropout_predict(
        model_tmp, sv_channels, target_size, device,
        n_samples=30, patch_width=256, stride=stride,
        dropout_p=p,
    )
    mc_results[p] = (m_prob, s_prob)
    print(f'p={p:.1f} — std range: [{s_prob.min():.6f}, {s_prob.max():.6f}], '
          f'mean std: {s_prob.mean():.6f}')
    del model_tmp

fig, axes = plt.subplots(1, len(dropout_rates), figsize=(6 * len(dropout_rates), 5),
                          sharex=True, sharey=True)
for ax, p in zip(axes, dropout_rates):
    _, s = mc_results[p]
    im = ax.imshow(s, aspect='auto', cmap='hot', origin='upper', extent=extent)
    ax.set_title(f'MC-Dropout std (p={p})')
    ax.set_xlabel('Ping')
    plt.colorbar(im, ax=ax, label='Std dev')

axes[0].set_ylabel('Range (m)')
plt.tight_layout()
plt.show()

## Prediction entropy

Binary entropy measures how far each pixel's probability is from a confident decision. Pixels near p=0.5 have maximum entropy (1.0), while pixels near 0 or 1 have entropy close to zero. This is complementary to MC-Dropout std — entropy captures uncertainty from the single deterministic prediction, while MC-Dropout captures model uncertainty from parameter sampling.

In [ ]:
entropy = prediction_entropy(prediction)

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharex=True, sharey=True)

axes[0].imshow(sv_channels[0], aspect='auto', cmap='viridis', vmin=0, vmax=1,
               origin='upper', extent=extent)
axes[0].set_title('Echogram (38 kHz)')
axes[0].set_ylabel('Range (m)')

im = axes[1].imshow(entropy, aspect='auto', cmap='inferno', vmin=0, vmax=1,
                     origin='upper', extent=extent)
axes[1].set_title('Prediction entropy')
plt.colorbar(im, ax=axes[1], label='H(p)')

# Overlay high-entropy contour on echogram
axes[0].contour(entropy, levels=[0.5], colors='red', linewidths=0.8,
                extent=extent, origin='upper')
axes[0].set_title('Echogram with high-entropy contour (H > 0.5)')

for ax in axes:
    ax.set_xlabel('Ping')

plt.tight_layout()
plt.show()

high_entropy_frac = (entropy > 0.5).mean()
print(f'{high_entropy_frac:.2%} of pixels have entropy > 0.5')

## Threshold sensitivity

The default classification threshold is 0.5, but the optimal operating point depends on the application. Sweeping thresholds from 0.05 to 0.95 shows how IoU, Dice, precision, and recall trade off, and how the bottom depth error changes. If the curves are flat around the peak, the model is robust to threshold choice; if they are steep, the threshold matters a lot.

In [ ]:
results = threshold_sensitivity(prediction, gt_mask, range_arr)
t = results['thresholds']

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(t, results['iou'], 'b-o', markersize=3, label='IoU')
ax1.plot(t, results['dice'], 'g-s', markersize=3, label='Dice')
ax1.plot(t, results['precision'], 'r--', alpha=0.7, label='Precision')
ax1.plot(t, results['recall'], 'm--', alpha=0.7, label='Recall')
ax1.axvline(0.5, color='gray', linestyle=':', alpha=0.5, label='t=0.5')

best_iou_idx = np.argmax(results['iou'])
ax1.axvline(t[best_iou_idx], color='blue', linestyle=':', alpha=0.5,
            label=f'Best IoU (t={t[best_iou_idx]:.2f})')
ax1.set_xlabel('Threshold')
ax1.set_ylabel('Score')
ax1.set_title('Segmentation metrics vs. threshold')
ax1.legend(fontsize=8)
ax1.grid(True, alpha=0.3)

ax2.plot(t, results['bottom_error_mean_m'], 'b-o', markersize=3, label='Mean')
ax2.plot(t, results['bottom_error_median_m'], 'g-s', markersize=3, label='Median')
ax2.axvline(0.5, color='gray', linestyle=':', alpha=0.5)
ax2.set_xlabel('Threshold')
ax2.set_ylabel('Bottom depth error (m)')
ax2.set_title('Depth error vs. threshold')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f'Best IoU: {results["iou"][best_iou_idx]:.4f} at threshold={t[best_iou_idx]:.2f}')
print(f'IoU at t=0.5: {results["iou"][9]:.4f}')

## Calibration

A well-calibrated model should have the property that when it predicts p=0.7, about 70% of those pixels are actually positive. The reliability diagram plots observed positive fraction against predicted probability per bin. The diagonal is perfect calibration. The Expected Calibration Error (ECE) summarises the deviation from the diagonal in a single number.

In [ ]:
bin_centers, observed, mean_pred, counts, ece = calibration_curve(
    prediction, gt_mask, n_bins=15,
)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Reliability diagram
ax1.plot([0, 1], [0, 1], 'k--', alpha=0.4, label='Perfect calibration')
ax1.bar(bin_centers, observed, width=1.0 / 15 * 0.8, alpha=0.6,
        color='steelblue', label='Observed freq')
ax1.plot(bin_centers, mean_pred, 'ro-', markersize=4, label='Mean pred')
ax1.set_xlabel('Predicted probability')
ax1.set_ylabel('Observed positive fraction')
ax1.set_title(f'Reliability diagram (ECE = {ece:.4f})')
ax1.legend()
ax1.set_xlim(0, 1)
ax1.set_ylim(0, 1)
ax1.grid(True, alpha=0.3)

# Bin histogram
ax2.bar(bin_centers, counts, width=1.0 / 15 * 0.8, color='gray', alpha=0.7)
ax2.set_xlabel('Predicted probability')
ax2.set_ylabel('Pixel count')
ax2.set_title('Prediction distribution')
ax2.set_yscale('log')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f'ECE: {ece:.4f}')
print(f'Total pixels: {counts.sum():,}')

## Bottom uncertainty band

Instead of a single bottom line, we can extract an uncertainty band from the probability map. The band spans the range where the probability transitions from 0.25 to 0.75, giving a visual sense of how sharply the model localises the seabed boundary. A narrow band means high confidence; a wide band means the model is unsure about the exact depth.

In [ ]:
def extract_bottom_line_robust(mask, threshold=0.5, min_run=5):
    """First row whose next `min_run` rows are also above threshold.

    Filters out thin spurious surface hits that fool the simple 'first
    crossing' rule under MC-Dropout perturbation.
    """
    binary = (mask >= threshold).astype(np.uint8)
    n_range, n_pings = binary.shape
    bottom_idx = np.full(n_pings, -1, dtype=np.int64)
    # cumulative-sum trick to count consecutive 1s starting at each row
    for ping in range(n_pings):
        col = binary[:, ping]
        # find rows where window of size min_run is all 1s
        if col.sum() < min_run:
            continue
        kernel = np.ones(min_run, dtype=np.int64)
        run = np.convolve(col, kernel, mode='valid')
        hits = np.where(run == min_run)[0]
        if len(hits) > 0:
            bottom_idx[ping] = hits[0]
    return bottom_idx


# Deterministic bottom line and probability-based uncertainty band
upper_depth, lower_depth, band_width = bottom_uncertainty_band(
    prediction, range_arr, lower_t=0.25, upper_t=0.75,
)

pred_bottom_idx = extract_bottom_line(prediction, threshold=0.5)
pred_bottom_depth = np.where(
    pred_bottom_idx >= 0,
    range_arr[np.clip(pred_bottom_idx, 0, len(range_arr) - 1)],
    np.nan,
)

# MC-Dropout bottom band — robust extraction (requires sustained detection)
bottom_depths_mc = np.full((all_preds.shape[0], n_pings), np.nan)
for i in range(all_preds.shape[0]):
    idx = extract_bottom_line_robust(all_preds[i], threshold=0.5, min_run=5)
    bottom_depths_mc[i] = np.where(
        idx >= 0, range_arr[np.clip(idx, 0, len(range_arr) - 1)], np.nan,
    )

mc_bottom_5  = np.nanpercentile(bottom_depths_mc, 15, axis=0)
mc_bottom_95 = np.nanpercentile(bottom_depths_mc, 85, axis=0)
mc_bottom_median = np.nanmedian(bottom_depths_mc, axis=0)

ping_axis = np.arange(n_pings)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), sharex=True,
                                gridspec_kw={'height_ratios': [3, 1]})

ax1.imshow(sv_channels[0], aspect='auto', cmap='viridis', vmin=0, vmax=1,
           origin='upper', extent=extent)
ax1.fill_between(ping_axis, mc_bottom_5, mc_bottom_95,
                 alpha=0.4, color='red', label=f'MC-Dropout 5–95th pct (p={DROPOUT_P})')
ax1.plot(ping_axis, pred_bottom_depth, 'r-', linewidth=0.8,
         label='Deterministic bottom (t=0.5)')
ax1.set_ylabel('Range (m)')
ax1.set_title(f'Echogram with MC-Dropout bottom band (p={DROPOUT_P}, n={N_SAMPLES})')
ax1.legend(loc='lower right', fontsize=8)

mc_band_width = np.abs(mc_bottom_95 - mc_bottom_5)
ax2.plot(ping_axis, mc_band_width, 'b-', linewidth=0.5)
ax2.fill_between(ping_axis, 0, mc_band_width, alpha=0.3, color='blue')
ax2.set_xlabel('Ping')
ax2.set_ylabel('Band width (m)')
ax2.set_title('MC-Dropout bottom uncertainty band width per ping')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

valid_bw = mc_band_width[~np.isnan(mc_band_width)]
print(f'MC band width — mean: {np.mean(valid_bw):.2f}m, '
      f'median: {np.median(valid_bw):.2f}m, '
      f'95th pct: {np.percentile(valid_bw, 95):.2f}m')

# Compare with deterministic probability band
print('\nDeterministic prob band (0.25–0.75):')
det_bw = band_width[~np.isnan(band_width)]
print(f'  mean: {np.mean(det_bw):.2f}m, median: {np.median(det_bw):.2f}m, '
      f'95th pct: {np.percentile(det_bw, 95):.2f}m')

## Findings

The analyses above give a first look at model confidence. MC-Dropout standard deviation and prediction entropy both highlight the seabed boundary as the primary region of uncertainty, which is expected — the water column and deep bottom are easy for the model, while the exact transition depth is harder. The threshold sensitivity sweep shows how robust the operating point is, and the calibration diagram reveals whether the raw probabilities can be trusted as confidence scores or whether post-hoc calibration (e.g. temperature scaling) might be worth exploring later. The uncertainty band provides a practical per-ping confidence interval that could be reported alongside the point estimate in downstream applications.